# 04 — Explore a fitted model and communicate outputs

This notebook fits a random forest to the full synthetic panel, computes training-data permutation importance, and maps mean fitted probabilities at illustrative point locations. Importance is measured on the same data used to fit the model, not on held-out folds. These are exploratory demonstration outputs, not validated geographic risk estimates. SHAP is a possible extension and is not implemented here.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from src import config
from src.models import fit_full
from src import viz

from src.features import load_panel
panel = load_panel()

## 1. Fit full model and compute permutation importance

In [2]:
rf, d = fit_full(panel, config.FULL_FEATURES, target=config.TARGET)
perm = permutation_importance(rf, d[config.FULL_FEATURES], d[config.TARGET],
                              scoring="roc_auc", n_repeats=30, random_state=42, n_jobs=-1)
viz.importance_plot(perm, config.FULL_FEATURES, config.FIGURES/"fig5_feature_importance.png")

## 2. Predicted county-level risk surface (mean monthly probability)

In [3]:
d = d.copy()
d["risk"] = rf.predict_proba(d[config.FULL_FEATURES])[:, 1]
risk = d.groupby("county").agg(lat=("lat","first"), lon=("lon","first"), risk=("risk","mean")).reset_index()
viz.risk_map(risk, config.FIGURES/"fig6_predicted_risk_map.png")

## Interpretation limits

The demonstration is synthetic. Training-data feature importance and fitted probabilities may be optimistic. The row bootstrap does not account for repeated counties, and county-grouped validation does not test future months. In an observational extension, surveillance effort, source quality, lag definitions, true geography, and residual dependence would require explicit evaluation.

## Conclusion and next steps

This prototype connects an ecological research question to reproducible feature engineering, logistic and random-forest modeling, county-grouped evaluation, and visual interpretation. It complements the observational dissertation project with Python ML implementation.

The next stage is to complete and validate real-data acquisition and assembly, compare matched predictor sets, evaluate temporal transfer and calibration, and use held-out interpretation and dependence-aware uncertainty before developing a user-facing decision-support tool.